# 03 - Batch Metadata Validation

Valida la extracción de metadata sobre un directorio completo de DICOMs.

**Qué se verifica:**
- `from_dicom` funciona correctamente para cada archivo del directorio.
- `MammographyMetadata` extrae sin errores: fabricante, modelo, lateralidad,
  vista, dimensiones, windowing y función VOI LUT.
- Los casos con error quedan recogidos en un DataFrame separado para análisis.

**Flujo:**
1. Configura `DATASET_DIR` y `MAX_FILES`.
2. Itera sobre los archivos encontrados.
3. Construye un DataFrame con los campos de metadata relevantes.
4. Muestra los primeros casos y, si los hay, la tabla de errores.

> Útil para detectar incompatibilidades al incorporar un nuevo dataset.

In [1]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'api_stable').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from api_stable.mammography import MammographyDicom

In [2]:
DATASET_DIR = Path('/home/eloygarcia/Escritorio/Datasets/Mammo-MX')
MAX_FILES = 50

if not DATASET_DIR.exists():
    raise FileNotFoundError(f'Actualiza DATASET_DIR: {DATASET_DIR}')

dicom_candidates = [p for p in DATASET_DIR.rglob('*') if p.is_file()]
dicom_candidates = dicom_candidates[:MAX_FILES]
print('Archivos a evaluar:', len(dicom_candidates))

Archivos a evaluar: 50


In [3]:
rows = []
errors = []

for path in dicom_candidates:
    try:
        mammo = MammographyDicom.from_dicom(path)
        md = mammo.metadata
        rows.append({
            'path': str(path),
            'manufacturer': md.vendor.manufacturer,
            'model_name': md.vendor.model_name,
            'laterality': md.breast.laterality,
            'view': md.breast.view,
            'rows': md.image.rows,
            'columns': md.image.columns,
            'photometric': md.image.photometric_interpretation,
        })
    except Exception as e:
        errors.append({'path': str(path), 'error': repr(e)})

df = pd.DataFrame(rows)
err_df = pd.DataFrame(errors)
print('OK:', len(df), ' | ERROR:', len(err_df))

OK: 49  | ERROR: 1


In [4]:
display(df.head(10))
if not err_df.empty:
    display(err_df.head(10))

,path,manufacturer,model_name,laterality,view,rows,columns,photometric
0,/home/eloygarcia/Escritorio/Datasets/Mammo-MX/...,"HOLOGIC, Inc.",Selenia Dimensions,L,MLO,3328,2560,MONOCHROME2
1,/home/eloygarcia/Escritorio/Datasets/Mammo-MX/...,"HOLOGIC, Inc.",Selenia Dimensions,R,CC,4096,3328,MONOCHROME2
2,/home/eloygarcia/Escritorio/Datasets/Mammo-MX/...,"HOLOGIC, Inc.",Selenia Dimensions,R,CC,4096,3328,MONOCHROME2
3,/home/eloygarcia/Escritorio/Datasets/Mammo-MX/...,"HOLOGIC, Inc.",Selenia Dimensions,R,CC,4096,3328,MONOCHROME2
4,/home/eloygarcia/Escritorio/Datasets/Mammo-MX/...,"HOLOGIC, Inc.",Selenia Dimensions,R,CC,4096,3328,MONOCHROME2
5,/home/eloygarcia/Escritorio/Datasets/Mammo-MX/...,"HOLOGIC, Inc.",Selenia Dimensions,R,MLO,4096,3328,MONOCHROME2
6,/home/eloygarcia/Escritorio/Datasets/Mammo-MX/...,"HOLOGIC, Inc.",Selenia Dimensions,L,CC,4096,3328,MONOCHROME2
7,/home/eloygarcia/Escritorio/Datasets/Mammo-MX/...,"HOLOGIC, Inc.",Selenia Dimensions,L,MLO,4096,3328,MONOCHROME2
8,/home/eloygarcia/Escritorio/Datasets/Mammo-MX/...,"HOLOGIC, Inc.",Selenia Dimensions,R,MLO,4096,3328,MONOCHROME2
9,/home/eloygarcia/Escritorio/Datasets/Mammo-MX/...,"HOLOGIC, Inc.",Selenia Dimensions,L,CC,4096,3328,MONOCHROME2


,path,error
0,/home/eloygarcia/Escritorio/Datasets/Mammo-MX/...,"InvalidDicomError(""File is missing DICOM File ..."
